*AI-generated draft (Claude, Anthropic) — for review. The labeling UI is version-controlled; the corrected boxes are your own ground truth. Click-added boxes use the typical worm size (per-frame median) — an approximation that is adequate for training recall.*

<span style="font-family: 'Courier New', monospace;">

# 31 · Box-labeler (click-to-mark) — correct pre-labels for the retrain

Each frame opens with the detector's **pre-labels** as orange boxes. You **click the missed worms** to add them.

**Kernel:** `joseph-scaleworm-thesis` (needs `ipympl`). Set **`UNIT`** in the setup cell to the batch you're correcting: `"2022"`, `"2021"` (both done), or **`"blurry_post2023"`** (Phase 2 — the current batch).

**Note for the `blurry_post2023` batch:** pre-labels were seeded by **v2 best.pt**, but it still only recovers ~12–14% of worms on blurry footage — so each frame starts with very few boxes (~2–3) and **you'll add most of the worms yourself.** That's expected: these hand-added boxes are exactly the training signal that teaches the next model to see worms in blurry frames.

**How to use (single clicks — same as the counter that worked)**
1. **Left-click each missed worm** → drops a box of the typical worm size on it.
2. **Right-click** on a box to delete it (a rare wrong pre-box, or a misclick).
3. **Undo** removes the last added box; **Clear** removes all.
4. **Save & Next ▶** writes the YOLO label to `labels/train/` + copies the image to `images/train/`, then loads the next frame. **Skip frame** advances without saving.
5. Zoom/pan with the toolbar, but **deselect zoom before clicking**.

Boxes for clicked worms are auto-sized to the frame's median worm box (good enough for training). Resumable; opens on the first un-corrected frame. When done, carve ~15% into `images/val`+`labels/val` (`scripts/make_val_split.py`), then run `python scripts/train_v2.py`.
</span>

In [1]:
%matplotlib widget
import shutil
import statistics
from pathlib import Path

import ipywidgets as widgets
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from IPython.display import display

REPO = Path("/home/jovyan/scaleworm-student-lab")
DS = REPO / "datasets/scaleworm_v2"
UNIT = "blurry_post2023"  # pre-label batch to correct: "2022", "2021", or "blurry_post2023"

PRE = DS / "prelabels" / UNIT
IMG_OUT = DS / "images/train"
LBL_OUT = DS / "labels/train"
IMG_OUT.mkdir(parents=True, exist_ok=True)
LBL_OUT.mkdir(parents=True, exist_ok=True)

DEFAULT_W, DEFAULT_H = 0.0342, 0.0585  # median normalized worm box (from -2022 pre-labels)

stems = sorted(p.stem for p in (PRE / "images").glob("*.png"))
done = sum(1 for s in stems if (LBL_OUT / f"{s}.txt").exists())
print(f"{len(stems)} {UNIT} frames to correct  ({done} already corrected).")
print("Left-click each missed worm; right-click a box to delete; then Save & Next.")


def load_yolo(path, W, H):
    boxes = []
    if path.exists():
        for line in path.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                cx, cy, w, h = (float(v) for v in p[1:])
                cx, cy, w, h = cx * W, cy * H, w * W, h * H
                boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return boxes


def save_yolo(path, boxes, W, H):
    lines = []
    for x1, y1, x2, y2 in boxes:
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        w, h = abs(x2 - x1) / W, abs(y2 - y1) / H
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    path.write_text("\n".join(lines) + ("\n" if lines else ""))

In [2]:
class BoxLabeler:
    """Correct pre-labels by single-clicking missed worms (mirrors the click-counter)."""

    def __init__(self, stems):
        self.stems = stems
        self.pos = next(
            (i for i, s in enumerate(stems) if not (LBL_OUT / f"{s}.txt").exists()), 0
        )
        self.boxes = []
        self.coll = None
        self.bw = DEFAULT_W
        self.bh = DEFAULT_H

        def mk(desc, style=""):
            return widgets.Button(description=desc, button_style=style,
                                  layout=widgets.Layout(width="auto"))

        self.b_undo = mk("Undo")
        self.b_clear = mk("Clear")
        self.b_prev = mk("◀ Prev")
        self.b_save = mk("Save & Next ▶", "success")
        self.b_skip = mk("Skip frame", "warning")
        self.b_undo.on_click(lambda _: self._undo())
        self.b_clear.on_click(lambda _: self._clear())
        self.b_prev.on_click(lambda _: self._prev())
        self.b_save.on_click(lambda _: self._save())
        self.b_skip.on_click(lambda _: self._advance())
        self.status = widgets.HTML()
        self.msg = widgets.Output()

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(12, 7))
        plt.ion()
        self.fig.canvas.header_visible = False
        self.fig.canvas.toolbar_position = "right"
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)

        controls = widgets.HBox(
            [self.b_undo, self.b_clear, self.b_prev, self.b_save, self.b_skip]
        )
        self.box = widgets.VBox([controls, self.status, self.fig.canvas, self.msg])
        self._load()

    def _stem(self):
        return self.stems[self.pos]

    def _load(self):
        stem = self._stem()
        img = mpimg.imread(PRE / "images" / f"{stem}.png")
        self.H, self.W = img.shape[:2]
        self.ax.clear()
        self.coll = None
        self.ax.imshow(img)
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        src = LBL_OUT / f"{stem}.txt"
        if not src.exists():
            src = PRE / "labels" / f"{stem}.txt"
        self.boxes = load_yolo(src, self.W, self.H)
        # per-frame median box size (fall back to the global default)
        if len(self.boxes) >= 3:
            self.bw = statistics.median((x2 - x1) / self.W for x1, _, x2, _ in self.boxes)
            self.bh = statistics.median((y2 - y1) / self.H for _, y1, _, y2 in self.boxes)
        else:
            self.bw, self.bh = DEFAULT_W, DEFAULT_H
        self._draw()
        self._status()

    def _draw(self):
        if self.coll is not None:
            try:
                self.coll.remove()
            except ValueError:
                pass
        rects = [mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1)
                 for x1, y1, x2, y2 in self.boxes]
        self.coll = PatchCollection(
            rects, facecolor="none", edgecolor="#D55E00", linewidths=1.6
        )
        self.ax.add_collection(self.coll)
        self.ax.set_title(f"{self._stem()}   —   {len(self.boxes)} boxes", fontsize=10)
        self.fig.canvas.draw_idle()

    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None:
            return
        if getattr(self.fig.canvas, "toolbar", None) and self.fig.canvas.toolbar.mode != "":
            return  # zoom/pan active
        if event.button == 1:  # add a median-sized box centred on the click
            w, h = self.bw * self.W, self.bh * self.H
            cx, cy = event.xdata, event.ydata
            self.boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
            self._draw()
        elif event.button == 3:  # delete the box under the cursor
            hit = [i for i, (x1, y1, x2, y2) in enumerate(self.boxes)
                   if x1 <= event.xdata <= x2 and y1 <= event.ydata <= y2]
            if hit:
                self.boxes.pop(hit[-1])
                self._draw()

    def _undo(self):
        if self.boxes:
            self.boxes.pop()
            self._draw()

    def _clear(self):
        self.boxes = []
        self._draw()

    def _status(self):
        self.status.value = (
            f"<b>Frame {self.pos + 1}/{len(self.stems)}</b> &nbsp; {self._stem()}"
        )

    def _save(self):
        stem = self._stem()
        save_yolo(LBL_OUT / f"{stem}.txt", self.boxes, self.W, self.H)
        shutil.copy(PRE / "images" / f"{stem}.png", IMG_OUT / f"{stem}.png")
        self._advance()

    def _advance(self):
        self.msg.clear_output()
        if self.pos < len(self.stems) - 1:
            self.pos += 1
            self._load()
        else:
            with self.msg:
                print("✅ All frames reviewed. Carve ~15% into images/val+labels/val, "
                      "then run scripts/train_v2.py")

    def _prev(self):
        if self.pos > 0:
            self.pos -= 1
            self._load()


app = BoxLabeler(stems)
display(app.box)